In [1]:
import os

classes = ['bear', 'fox', 'reindeer', 'santa', 'cyclist', 'robot']
splits = ['train', 'val']

for split in splits:
    for cls in classes:
        path = f'dataset/{split}/{cls}'
        os.makedirs(path, exist_ok=True)
        print(f'Created: {path}')

print('\nAll folders ready!')

Created: dataset/train/bear
Created: dataset/train/fox
Created: dataset/train/reindeer
Created: dataset/train/santa
Created: dataset/train/cyclist
Created: dataset/train/robot
Created: dataset/val/bear
Created: dataset/val/fox
Created: dataset/val/reindeer
Created: dataset/val/santa
Created: dataset/val/cyclist
Created: dataset/val/robot

All folders ready!


In [ ]:
import cv2
from flask import Flask, Response
from picamera2 import Picamera2
from ultralytics import YOLO  # <-- 1. Import YOLO

app = Flask(__name__)

# <-- 2. Load your model here (ensure best.pt is on the Pi in the same folder!)
model = YOLO("best.pt")

picam2 = Picamera2()
config = picam2.create_video_configuration(main={"size": (320, 240)})
picam2.configure(config)
picam2.start()

def gen_frames():
    while True:
        # Grab the raw frame from the camera
        frame = picam2.capture_array()

        # strip alpha channel
        frame = frame[:, :, :3]

        # <-- 3. Pass the raw frame to your model
        # verbose=False prevents the terminal from printing stats 30 times a second
        results = model.predict(source=frame, conf=0.25, imgsz=320, verbose=False)

        # <-- 4. Draw the bounding boxes!
        # results[0].plot() returns a numpy array of the image with the boxes applied
        annotated_frame = results[0].plot()

        # flip the colors to look right
        annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
        # Encode the ANNOTATED frame (not the raw frame) into the JPEG buffer
        ret, buffer = cv2.imencode('.jpg', annotated_frame)

        yield (b'--frame\r\n'
            b'Content-Type: image/jpeg\r\n\r\n' + buffer.tobytes() + b'\r\n')

@app.route('/')
def video_feed():
    return Response(gen_frames(), mimetype='multipart/x-mixed-replace; boundary=frame')

if __name__ == '__main__':
    try:
        # Run the Flask server
        app.run(host='0.0.0.0', port=5000)
    except KeyboardInterrupt:
        # This catches the Ctrl+C
        pass
    finally:
        # This block will ALWAYS run when the script stops,
        # ensuring the camera is cleanly released.
        print("\nReleasing camera resources...")
        picam2.stop()
        picam2.close()
        print("Shutdown complete.")